In [ ]:
# ## Step 0: Load Data & One-Hot Encode
# We load the dataset, convert categorical features to dummy variables,
# and fill missing values (mostly created during encoding).
from sklearn.preprocessing import RobustScaler, PolynomialFeatures
from sklearn.linear_model import ElasticNet
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, train_test_split
import pandas as pd
import numpy as np

data = pd.read_csv("analysis_data.csv")
data_encoded = pd.get_dummies(data, drop_first=True)
data_encoded = data_encoded.fillna(0)


# ## Step 1: Remove Outliers with IQR
# Outliers affected model stability, so the IQR technique was used.
def remove_outliers_iqr(df):
    df_clean = df.copy()
    numeric_cols = df_clean.select_dtypes(include=['int64','float64']).columns
    
    for col in numeric_cols:
        Q1 = df_clean[col].quantile(0.25)
        Q3 = df_clean[col].quantile(0.75)
        IQR = Q3 - Q1
        lower = Q1 - 1.5 * IQR
        upper = Q3 + 1.5 * IQR
        df_clean = df_clean[(df_clean[col] >= lower) & (df_clean[col] <= upper)]
    return df_clean

data_clean = remove_outliers_iqr(data_encoded)


# ## Step 2: Train–Test Split
# Standard 80/20 split. Target = monthly_spend.
y = data_clean['monthly_spend']
X = data_clean.drop(columns=['monthly_spend'])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


# ## Step 3: Pipeline (Scaling → Polynomial → ElasticNet)
# RobustScaler handles outliers, Polynomial captures interactions,
# ElasticNet provides balanced regularization.
pipe = Pipeline([
    ('scaler', RobustScaler()),
    ('poly', PolynomialFeatures(include_bias=False)),
    ('model', ElasticNet(max_iter=3000))
])


# ## Step 4: Hyperparameter Tuning (GridSearchCV)
# Search space includes polynomial degree, alpha, and l1_ratio.
param_grid = {
    'poly__degree': [1, 2],
    'model__alpha': [0.01, 0.1, 0.5],
    'model__l1_ratio': [0.1, 0.3, 0.5]
}

grid = GridSearchCV(
    pipe,
    param_grid,
    scoring='neg_root_mean_squared_error',
    cv=5,
    n_jobs=-1
)

grid.fit(X_train, y_train)

print("Best RMSE (CV):", -grid.best_score_)
print("Best Params:", grid.best_params_)


# ## Step 5: Evaluate on Test Set
# Final evaluation to measure generalization performance.
from sklearn.metrics import mean_squared_error

pred = grid.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, pred))
print("Test RMSE:", rmse)

# ElasticNet performed best because:
# - Good balance between complexity and regularization
# - Polynomial degree 2 captured mild nonlinearity
# - RobustScaler stabilized training